In [72]:
! pip install squidpy

In [73]:
import pandas as pd
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix

spot_coord = pd.read_csv("/data2/r10user3/Spatial_Gene_Cell_Ratio/data/human_lung_cell2location/WSA_LngSP8759312/spots.csv", index_col=0)
spot_coord['X'] = spot_coord['X'] / 112
spot_coord['Y'] = spot_coord['Y'] / 112

cell_abundance = pd.read_csv("/data2/r10user3/Spatial_Gene_Cell_Ratio/data/human_lung_cell2location/WSA_LngSP8759312/cell_ratio.csv", index_col=0)
cell_abundance = cell_abundance.loc[spot_coord.index]

location = np.matrix(spot_coord.values)
Xdense = np.matrix(cell_abundance.values)

X = csr_matrix(Xdense, dtype=np.float32)
adata = ad.AnnData(X, obsm={"spatial": location})
adata

AnnData object with n_obs × n_vars = 2564 × 80
    obsm: 'spatial'

In [74]:
adata.obs_names = list(cell_abundance.index)
adata.var_names = list(cell_abundance.columns)
adata

AnnData object with n_obs × n_vars = 2564 × 80
    obsm: 'spatial'

In [75]:
import squidpy as sq

sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial")
adata

/home/r10user3/anaconda3/envs/pyg/lib/python3.8/site-packages/sklearn/utils/validation.py:585: FutureWarning: np.matrix usage is deprecated in 1.0 and will raise a TypeError in 1.2. Please convert to a numpy array with np.asarray. For more information see: https://numpy.org/doc/stable/reference/generated/numpy.matrix.html
  warnings.warn(


AnnData object with n_obs × n_vars = 2564 × 80
    uns: 'spatial_neighbors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'

In [76]:
sq.gr.spatial_autocorr(adata, mode="moran")

In [77]:
adata.uns["moranI"].head(10)

,I,pval_norm,var_norm,pval_norm_fdr_bh
q05cell_abundance_w_sf_NK_CD16hi,0.812455,0.0,0.00013,0.0
q05cell_abundance_w_sf_NK_CD11d,0.802487,0.0,0.00013,0.0
q05cell_abundance_w_sf_CD4_TRM,0.799560,0.0,0.00013,0.0
q05cell_abundance_w_sf_gdT,0.795093,0.0,0.00013,0.0
q05cell_abundance_w_sf_CD4_EM_Effector,0.794764,0.0,0.00013,0.0
q05cell_abundance_w_sf_ILC,0.794412,0.0,0.00013,0.0
q05cell_abundance_w_sf_CD8_EM,0.789481,0.0,0.00013,0.0
q05cell_abundance_w_sf_SMG_Duct,0.786928,0.0,0.00013,0.0
q05cell_abundance_w_sf_T_reg,0.783730,0.0,0.00013,0.0
q05cell_abundance_w_sf_B_naive,0.780201,0.0,0.00013,0.0


In [78]:
df  = pd.DataFrame(adata.uns["moranI"])
df

,I,pval_norm,var_norm,pval_norm_fdr_bh
q05cell_abundance_w_sf_NK_CD16hi,0.812455,0.000000e+00,0.00013,0.000000e+00
q05cell_abundance_w_sf_NK_CD11d,0.802487,0.000000e+00,0.00013,0.000000e+00
q05cell_abundance_w_sf_CD4_TRM,0.799560,0.000000e+00,0.00013,0.000000e+00
q05cell_abundance_w_sf_gdT,0.795093,0.000000e+00,0.00013,0.000000e+00
q05cell_abundance_w_sf_CD4_EM_Effector,0.794764,0.000000e+00,0.00013,0.000000e+00
...,...,...,...,...
q05cell_abundance_w_sf_Chondrocyte,0.268113,0.000000e+00,0.00013,0.000000e+00
q05cell_abundance_w_sf_Macro_CHIT1,0.195916,0.000000e+00,0.00013,0.000000e+00
q05cell_abundance_w_sf_Fibro_alveolar,0.147083,0.000000e+00,0.00013,0.000000e+00
q05cell_abundance_w_sf_AT1,0.132477,0.000000e+00,0.00013,0.000000e+00


In [79]:
# load multi-task GNN cell pearson list
import pickle

with open('/data2/r10user3/Spatial_Gene_Cell_Ratio/code/pearson_log/cell_gene_2layersimpleGNN_lr1e-4_cell_pearson_list.pkl', 'rb') as f:
    multi_cell_pearson_list = pickle.load(f)
df['pearson'] = multi_cell_pearson_list
df

,I,pval_norm,var_norm,pval_norm_fdr_bh,pearson
q05cell_abundance_w_sf_NK_CD16hi,0.812455,0.000000e+00,0.00013,0.000000e+00,0.164479
q05cell_abundance_w_sf_NK_CD11d,0.802487,0.000000e+00,0.00013,0.000000e+00,0.140642
q05cell_abundance_w_sf_CD4_TRM,0.799560,0.000000e+00,0.00013,0.000000e+00,0.313251
q05cell_abundance_w_sf_gdT,0.795093,0.000000e+00,0.00013,0.000000e+00,0.418429
q05cell_abundance_w_sf_CD4_EM_Effector,0.794764,0.000000e+00,0.00013,0.000000e+00,0.358119
...,...,...,...,...,...
q05cell_abundance_w_sf_Chondrocyte,0.268113,0.000000e+00,0.00013,0.000000e+00,0.241010
q05cell_abundance_w_sf_Macro_CHIT1,0.195916,0.000000e+00,0.00013,0.000000e+00,0.482241
q05cell_abundance_w_sf_Fibro_alveolar,0.147083,0.000000e+00,0.00013,0.000000e+00,0.379403
q05cell_abundance_w_sf_AT1,0.132477,0.000000e+00,0.00013,0.000000e+00,0.468492


In [82]:
df.sort_values('I', ascending=False).to_csv('/data2/r10user3/Spatial_Gene_Cell_Ratio/code/raw_data/MoranI.csv')